# Teacher server: phi3.5 on Ollama (free Colab T4) exposed through a Cloudflare quick tunnel

The k3d worker pods (namespace `pipeline`) call this server via `OLLAMA_URL`. Labelling 3,000 commands on a laptop CPU takes hours; a free T4 does it in well under an hour.
**Runtime -> Change runtime type -> T4 GPU**, then run the cells in order and keep this tab open while `make label` runs. Settings (temperature 0, seed 42, prompt) are sent by the workers with every request, not set here.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_NUM_PARALLEL='3', OLLAMA_KEEP_ALIVE='-1')
ollama = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434/api/version

In [ ]:
!ollama pull phi3.5
!ollama list
!ollama show phi3.5 | head -20

In [ ]:
# sanity check with the fixed settings
import requests, json
r = requests.post('http://localhost:11434/api/generate', json={
    'model': 'phi3.5', 'stream': False, 'format': 'json',
    'options': {'temperature': 0, 'seed': 42, 'num_predict': 128},
    'prompt': 'Return the JSON frame (intent, date, time, place) for: set an alarm for six am tomorrow'})
print(r.json()['response'], r.json().get('eval_duration', 0) / 1e9, 's')

In [ ]:
# public HTTPS URL for the cluster workers (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/download/2024.11.1/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:11434',
                           '--http-host-header', 'localhost:11434'], stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m: url = m.group(0); break
    time.sleep(1)
print('OLLAMA_URL =', url)
time.sleep(5)
!curl -s {url}/api/version

In [ ]:
# keep the session busy and show progress (stop this cell when labelling is done)
while True:
    print(time.strftime('%H:%M:%S'), subprocess.run(['bash', '-c', 'grep -c "POST" ollama.log'], capture_output=True, text=True).stdout.strip(), 'requests served')
    time.sleep(60)